# 雨の日は、本当に気温が低いのか？

**氏名：三橋龍之輔**  
**第3週課題｜データで見つけた、小さな発見**

## 1．調べたいこと

雨の日は涼しいイメージがあるけれど、本当にそうなのか気になった。そこで、**2026年9月の柏周辺**について、雨が降った日と降らなかった日の最高気温を比べる。

- **問い：** 雨が降った日の方が、雨が降らなかった日より日最高気温は低いのか？
- **雨の日の決め方：** その日の合計降水量が **1.0 mm以上** の日。1.0 mm未満の日を「雨が少ない日」として比較する。
- **調べる範囲：** 2026年9月1日〜9月30日（日本時間）。

※ 雨が気温を下げるという因果関係を証明するものではなく、同じ月のデータの**傾向**を調べる。

## 2．使うデータと出典

公開API **Open-Meteo Historical Weather API** の過去天気データを使う。APIの返す `daily.temperature_2m_max`（日最高気温、℃）と `daily.precipitation_sum`（日降水量、mm）を読み込む。

- **出典URL：** https://open-meteo.com/en/docs/historical-weather-api
- **対象地点：** 千葉県柏市周辺（北緯35.86度、東経139.97度）
- **期間：** 2026年9月1日〜9月30日
- **取得日：** 実行時に下のコードが自動表示する
- **注意：** 過去の天気についてのモデル・再解析データであり、柏市内の観測所で測定した実測値そのものとは限らない。

APIの取得URLも下に表示する。

In [1]:
import json
from urllib.request import urlopen
from urllib.parse import urlencode
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo

parameters = {
    "latitude": 35.86,
    "longitude": 139.97,
    "start_date": "2026-09-01",
    "end_date": "2026-09-30",
    "daily": "temperature_2m_max,precipitation_sum",
    "timezone": "Asia/Tokyo"
}
api_url = "https://archive-api.open-meteo.com/v1/archive?" + urlencode(parameters)
fetched_at = datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M JST")

try:
    with urlopen(api_url, timeout=30) as response:
        data = json.load(response)
except Exception as error:
    raise RuntimeError("天気データを取得できませんでした。インターネット接続を確認して再実行してください。") from error

if data.get("error"):
    raise RuntimeError("APIがエラーを返しました: " + str(data.get("reason")))

print("取得日時:", fetched_at)
print("APIのURL:", api_url)
print("タイムゾーン:", data.get("timezone"))

取得日時: 2026-10-19 10:22 JST
APIのURL: https://archive-api.open-meteo.com/v1/archive?latitude=35.86&longitude=139.97&start_date=2026-09-01&end_date=2026-09-30&daily=temperature_2m_max%2Cprecipitation_sum&timezone=Asia%2FTokyo
タイムゾーン: Asia/Tokyo


## 3．データを読み込む

各日付の最高気温と降水量を対応させる。数値が欠けている日は比較から外す。

In [2]:
daily = data["daily"]
dates = daily["time"]
max_temps = daily["temperature_2m_max"]
precipitations = daily["precipitation_sum"]

assert len(dates) == len(max_temps) == len(precipitations), "データの長さが一致しません"
records = []
for date, temp, rain in zip(dates, max_temps, precipitations):
    if temp is None or rain is None:
        continue
    records.append({"date": date, "temp": float(temp), "rain": float(rain)})

if not records:
    raise RuntimeError("比較できる気温・降水量データがありません")

print(f"取得できた日数: {len(records)}日")
print("先頭3日分（APIから読み込んだ値）:")
for day in records[:3]:
    print(f"{day['date']}: 最高気温 {day['temp']:.1f}℃、降水量 {day['rain']:.1f}mm")

取得できた日数: 30日
先頭3日分（APIから読み込んだ値）:
2026-09-01: 最高気温 27.5℃、降水量 0.7mm
2026-09-02: 最高気温 31.0℃、降水量 0.3mm
2026-09-03: 最高気温 30.5℃、降水量 3.0mm


## 4．雨が降った日と雨が少ない日を比べる

まず日数を数えて、グループごとの平均最高気温を計算する。雨の日の方が涼しいと決めつけず、**計算結果を見てから判断**する。

In [3]:
rainy = [r for r in records if r["rain"] >= 1.0]
less_rain = [r for r in records if r["rain"] < 1.0]

print(f"雨の日（1.0 mm以上）: {len(rainy)}日")
print(f"雨が少ない日（1.0 mm未満）: {len(less_rain)}日")

if not rainy or not less_rain:
    raise RuntimeError("片方のグループが0日です。比較期間や雨の基準を変更してください。")

avg_rainy = sum(r["temp"] for r in rainy) / len(rainy)
avg_less = sum(r["temp"] for r in less_rain) / len(less_rain)
difference = avg_rainy - avg_less

print(f"雨の日の平均最高気温: {avg_rainy:.1f}℃")
print(f"雨が少ない日の平均最高気温: {avg_less:.1f}℃")
print(f"差（雨の日 − 雨が少ない日）: {difference:+.1f}℃")

雨の日（1.0 mm以上）: 24日
雨が少ない日（1.0 mm未満）: 6日
雨の日の平均最高気温: 24.8℃
雨が少ない日の平均最高気温: 27.8℃
差（雨の日 − 雨が少ない日）: -3.0℃


## 5．グラフで違いを見てみる

**棒の高さは平均最高気温**。見比べやすいように、0℃から表示する。追加のライブラリを使わず、Notebook上でグラフを描く。

In [4]:
from IPython.display import HTML, display

values = [("雨の日（1mm以上）", avg_rainy, "#2776c5"),
          ("雨が少ない日", avg_less, "#e89a31")]
scale_max = max(35.0, max(v for _, v, _ in values) + 3)

bars = ""
for label, value, color in values:
    bars += (f'<div style="margin:14px 0;font-size:14px">{label}：<b>{value:.1f}℃</b>'
             f'<div style="height:24px;background:#edf0f4;border-radius:4px;margin-top:6px">'
             f'<div style="height:100%;width:{100*value/scale_max:.1f}%;background:{color};border-radius:4px"></div>'
             f'</div></div>')
display(HTML('<div style="max-width:640px;padding:18px;border:1px solid #ddd;'
             'border-radius:8px;color:#202938;background:white">'
             '<b>9月の平均日最高気温（柏周辺）</b>' + bars +
             f'<div style="font-size:12px;color:#555">0℃から {scale_max:.0f}℃までの同一目盛。'
             '降水量1.0mmを境に分類。</div></div>'))

## 6．元データで確認する

計算だけで終わらせず、雨が降った日の例と雨が少なかった日の例をそれぞれ表示して、APIの値と照らし合わせる。最初の一覧と同じ `records` を使用する。

In [5]:
for group_name, group in [("雨の日", rainy), ("雨が少ない日", less_rain)]:
    print("■", group_name, "の例（最大3日）")
    for row in group[:3]:
        idx = dates.index(row["date"])
        assert row["temp"] == max_temps[idx] and row["rain"] == precipitations[idx]
        print(f"  {row['date']}: 降水量 {row['rain']:.1f}mm / 最高気温 {row['temp']:.1f}℃")
    print()
print("元データとの照合: OK（表示した値がAPIの配列と一致）")

■ 雨の日 の例（最大3日）
  2026-09-03: 降水量 3.0mm / 最高気温 30.5℃
  2026-09-04: 降水量 21.5mm / 最高気温 25.2℃
  2026-09-05: 降水量 12.3mm / 最高気温 26.0℃

■ 雨が少ない日 の例（最大3日）
  2026-09-01: 降水量 0.7mm / 最高気温 27.5℃
  2026-09-02: 降水量 0.3mm / 最高気温 31.0℃
  2026-09-13: 降水量 0.6mm / 最高気温 26.4℃

元データとの照合: OK（表示した値がAPIの配列と一致）


## 7．発見したこと・確かめたこと・限界

以下は**APIから取得した値によって自動で作成する結果**。数値や結論を手で決め打ちしていないので、APIの内容が変わっても計算と文章が一致する。

In [6]:
if difference < -0.05:
    finding = "雨の日の方が平均最高気温が低かった"
elif difference > 0.05:
    finding = "雨の日の方が平均最高気温が高かった"
else:
    finding = "両グループの平均最高気温はほぼ同じだった"

report = f"""### 調べて分かったこと

2026年9月の柏周辺を調べると、降水量が1.0mm以上の日は **{len(rainy)}日**、
1.0mm未満の日は **{len(less_rain)}日** あった。
平均最高気温はそれぞれ **{avg_rainy:.1f}℃** と **{avg_less:.1f}℃** で、
その差（雨の日－雨が少ない日）は **{difference:+.1f}℃** だった。
今回のデータでは、**{finding}**。

### 元データで確かめたこと

APIの `daily` に入っている日付・日最高気温・日降水量を同じ順番で対応させ、
各グループの件数と平均をPythonで計算した。さらに、両グループから日付を取り出して
元の配列の値と照らし合わせ、一致していることを確認した。

### このデータだけでは分からないこと

これは**柏周辺の1か月分の傾向**で、雨が気温を下げたと断定はできない。
季節の移り変わりや天気の別の要素も関わる。また、1.0mmという分類の基準を変えると
結果が変わる可能性がある。過去天気APIの値は地点周辺のモデル・再解析データであり、
現地の観測値と完全に同じとは限らない。

### 出典・取得日

- 出典：Open-Meteo Historical Weather API
- ドキュメント：https://open-meteo.com/en/docs/historical-weather-api
- 取得日時：**{fetched_at}**
- 対象期間：2026年9月1日〜9月30日（日本時間）
- APIリクエストURL：{api_url}

### AIの利用

ChatGPTを使って、Pythonコードの作成と文章の整理を手伝ってもらった。
実行結果と元データの一致はNotebook内で確認した。
"""
from IPython.display import Markdown
display(Markdown(report))

### 調べて分かったこと

2026年9月の柏周辺を調べると、降水量が1.0mm以上の日は **24日**、
1.0mm未満の日は **6日** あった。
平均最高気温はそれぞれ **24.8℃** と **27.8℃** で、
その差（雨の日－雨が少ない日）は **-3.0℃** だった。
今回のデータでは、**雨の日の方が平均最高気温が低かった**。

### 元データで確かめたこと

APIの `daily` に入っている日付・日最高気温・日降水量を同じ順番で対応させ、
各グループの件数と平均をPythonで計算した。さらに、両グループから日付を取り出して
元の配列の値と照らし合わせ、一致していることを確認した。

### このデータだけでは分からないこと

これは**柏周辺の1か月分の傾向**で、雨が気温を下げたと断定はできない。
季節の移り変わりや天気の別の要素も関わる。また、1.0mmという分類の基準を変えると
結果が変わる可能性がある。過去天気APIの値は地点周辺のモデル・再解析データであり、
現地の観測値と完全に同じとは限らない。

### 出典・取得日

- 出典：Open-Meteo Historical Weather API
- ドキュメント：https://open-meteo.com/en/docs/historical-weather-api
- 取得日時：**2026-10-19 10:22 JST**
- 対象期間：2026年9月1日〜9月30日（日本時間）
- APIリクエストURL：https://archive-api.open-meteo.com/v1/archive?latitude=35.86&longitude=139.97&start_date=2026-09-01&end_date=2026-09-30&daily=temperature_2m_max%2Cprecipitation_sum&timezone=Asia%2FTokyo

### AIの利用

ChatGPTを使って、Pythonコードの作成と文章の整理を手伝ってもらった。
実行結果と元データの一致はNotebook内で確認した。


## 8．提出前の確認

1. 上の **「すべてを実行（Run All）」** を押し、エラーがないことを確認する。
2. **グラフ・APIから取得した値・「調べて分かったこと」の文章が表示された状態**で保存する（`Ctrl + S`）。
3. 先生の指示にある `.ipynb` をUNIPAに提出する。このNotebookはAPIからデータを直接読むため、別途CSV・JSONファイルを作っていない。

**注：** Internet接続できない場合は、取得エラーが表示される。エラーが残った状態では提出せず、接続環境を確認してから再実行する。